## Project: Implementing a ChatGPT App with LangChain from Scratch



In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [4]:
from langchain_openai import ChatOpenAI

# Updated: SystemMessage moved from 'langchain.schema' to 'langchain_core.messages'
# Old: from langchain.schema import SystemMessage
from langchain_core.messages import SystemMessage

# Updated: ChatPromptTemplate and HumanMessagePromptTemplate moved from
# 'langchain.prompts' to 'langchain_core.prompts'
# Old: from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

# Updated: StrOutputParser import unchanged (already from langchain_core)
from langchain_core.output_parsers import StrOutputParser

# Note: LLMChain (from langchain.chains) is no longer needed here.
# The chain below uses the modern LCEL syntax (prompt | llm | parser) instead.

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=1)

prompt = ChatPromptTemplate(
    input_variables=["content"],
    messages=[
#       SystemMessage(content="You are a chatbot having a conversation with a human."),
        SystemMessage(content='You respond only in German.'),
        HumanMessagePromptTemplate.from_template("{content}")
    ]
)

# Updated: replaced LLMChain with LCEL pipe syntax (prompt | llm | parser)
# Old:
# chain = LLMChain(llm=llm, prompt=prompt, verbose=True)
chain = prompt | llm | StrOutputParser()

while True:
    content = input('Your prompt: ')
    if content.lower() in ['quit', 'q', 'exit', 'bye']:
        print('Goodbye!')
        break

    response = chain.invoke({'content': content})
    print(response)
    print('-' * 50)


Es gibt mehr Mikroorganismen in deinem Körper als Zellen.
--------------------------------------------------
Goodbye!


## Adding Chat Memory Using ConversationBufferMemory

In [7]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser

# Updated: ChatPromptTemplate, HumanMessagePromptTemplate, and MessagesPlaceholder
# moved from 'langchain.prompts' to 'langchain_core.prompts'
# Old: from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

# Updated: LLMChain + ConversationBufferMemory replaced with RunnableWithMessageHistory (LCEL)
# Old: from langchain.chains import LLMChain
# Old: from langchain.memory import ConversationBufferMemory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model_name='gpt-5.2', temperature=0.5)

# 1. Create in-memory session store (replaces ConversationBufferMemory)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 2. Build prompt with MessagesPlaceholder for chat history (unchanged)
prompt = ChatPromptTemplate(
    input_variables=["content", "chat_history"],
    messages=[
        SystemMessage(content="You are a chatbot having a conversation with a human."),
        MessagesPlaceholder(variable_name="chat_history"),  # Where the memory will be stored.
        HumanMessagePromptTemplate.from_template("{content}")
    ]
)

# 3. Wrap the LCEL chain with RunnableWithMessageHistory (replaces LLMChain with memory=)
# Old:
# chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=False)
chain = RunnableWithMessageHistory(
    prompt | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="content",
    history_messages_key="chat_history",
)

while True:
    content = input('Your prompt: ')
    if content.lower() in ['quit', 'exit', 'bye']:
        print('Goodbye!')
        break

    response = chain.invoke(
        {'content': content},
        config={"configurable": {"session_id": "default"}}
    )
    print(response)
    print('-' * 50)


Here are 10 standout tourist destinations to consider for 2026 (mix of cities, nature, and culture across regions):

1. **Tokyo, Japan** – Food, neighborhoods, day trips (Hakone/Nikko), and cutting-edge culture.  
2. **Kyoto, Japan** – Temples, gardens, traditional districts, and seasonal scenery.  
3. **Paris, France** – Classic museums, neighborhoods, day trips (Versailles/Champagne).  
4. **Rome, Italy** – Ancient history, Vatican City, and unbeatable dining.  
5. **London, UK** – Theater, museums, markets, and easy rail trips around England.  
6. **Barcelona, Spain** – Architecture (Gaudí), beaches, nightlife, and great food.  
7. **Dubai, UAE** – Modern skyline, desert experiences, luxury shopping, big attractions.  
8. **Bali, Indonesia** – Beaches, surf, temples, wellness, and nearby islands.  
9. **Cape Town, South Africa** – Table Mountain, beaches, wine country, and wildlife access.  
10. **Queenstown, New Zealand** – Adventure hub with spectacular landscapes (Fiordland nearb

## Saving Chat Sessions

In [8]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser

# Updated: ChatPromptTemplate, HumanMessagePromptTemplate, and MessagesPlaceholder
# moved from 'langchain.prompts' to 'langchain_core.prompts'
# Old: from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

# Updated: LLMChain + ConversationBufferMemory replaced with RunnableWithMessageHistory (LCEL)
# Old: from langchain.chains import LLMChain
# Old: from langchain.memory import ConversationBufferMemory, FileChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Import FileChatMessageHistory
# Updated: FileChatMessageHistory moved from 'langchain.memory' to 'langchain_community.chat_message_histories'
# Old: from langchain.memory import ConversationBufferMemory, FileChatMessageHistory
from langchain_community.chat_message_histories import FileChatMessageHistory

llm = ChatOpenAI(model_name='gpt-5-nano', temperature=0)

# 2. Use FileChatMessageHistory directly as the session history provider
# Old:
# history = FileChatMessageHistory('chat_history.json')
# memory = ConversationBufferMemory(memory_key='chat_history', chat_memory=history, return_messages=True)
def get_session_history(session_id: str):
    return FileChatMessageHistory(f'{session_id}.json')

prompt = ChatPromptTemplate(
    input_variables=["content", "chat_history"],
    messages=[
        SystemMessage(content="You are a chatbot having a conversation with a human."),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{content}")
    ]
)

# 3. Wrap the LCEL chain with RunnableWithMessageHistory (replaces LLMChain with memory=)
# Old:
# chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=False)
chain = RunnableWithMessageHistory(
    prompt | llm | StrOutputParser(),
    get_session_history,
    input_messages_key="content",
    history_messages_key="chat_history",
)

while True:
    content = input('Your prompt: ')
    if content.lower() in ['quit', 'exit', 'bye']:
        print('Goodbye!')
        break

    response = chain.invoke(
        {'content': content},
        config={"configurable": {"session_id": "chat_history"}}
    )
    print(response)
    print('-' * 50)


By the year 2500, Earth has largely stabilized its climate thanks to abundant clean energy, large-scale ecosystem restoration, and resilience-focused governance. Cities rise as vertical megacities and floating districts linked by rapid, low-emission transport, all woven into continuous green corridors and biodiversity reserves. Energy flows from fusion, solar, and other clean sources, powering resilient grids and climate-adapted infrastructure. People live in communities that prioritize food security, urban farming, and harmonious coexistence with nature, with oceans and forests largely restored. Global collaboration, education, and curiosity drive science and culture, making Earth a durable, interconnected home with a shared sense of stewardship.
--------------------------------------------------
In a stabilized Earth of 2500, the highest risk is systemic governance and resilience risk: if global or regional cooperation erodes, the ability to maintain clean energy systems, climate res

In [9]:
# The messages property contains the list of messages in order.
# Updated: 'history' variable replaced by get_session_history() from the cell above
# Old: print(history.messages)
print(get_session_history("chat_history").messages)


[HumanMessage(content='Tell about Earth in 2500 in a few sentences.', additional_kwargs={}, response_metadata={}), AIMessage(content='By the year 2500, Earth has largely stabilized its climate thanks to abundant clean energy, large-scale ecosystem restoration, and resilience-focused governance. Cities rise as vertical megacities and floating districts linked by rapid, low-emission transport, all woven into continuous green corridors and biodiversity reserves. Energy flows from fusion, solar, and other clean sources, powering resilient grids and climate-adapted infrastructure. People live in communities that prioritize food security, urban farming, and harmonious coexistence with nature, with oceans and forests largely restored. Global collaboration, education, and curiosity drive science and culture, making Earth a durable, interconnected home with a shared sense of stewardship.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='w